# load_openaire_researchproduct_collectedfrom

Prototipo del nodo `load_openaire_researchproduct_collectedfrom` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_collectedfrom(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_collectedfrom = df[['id', 'collectedFrom', *_EXTRACTED_META_COLS]].explode('collectedFrom').reset_index(drop=True)
    df_research_collectedfrom.rename(columns={'id':'researchproduct_id'}, inplace=True)

    df_collectedfrom = pd.json_normalize(df_research_collectedfrom['collectedFrom'])
    df_collectedfrom.rename(columns={'key':'datasource_id'}, inplace=True)

    df_research_collectedfrom = pd.concat(
        [
            df_research_collectedfrom[['researchproduct_id', *_EXTRACTED_META_COLS]].reset_index(drop=True),
            df_collectedfrom.loc[:,['datasource_id','value']].reset_index(drop=True),
        ],
        axis=1
    )

    df_research_collectedfrom = _add_openaire_loaded_metadata(df_research_collectedfrom)

    return df_research_collectedfrom


In [ ]:
df_research_collectedfrom = load_openaire_researchproduct_collectedfrom(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_research_collectedfrom', 'rows': len(df_research_collectedfrom), 'columns': len(df_research_collectedfrom.columns)}])


In [ ]:
df_research_collectedfrom.head(2)
